In [ ]:
# Data source switch
#   USE_RAW = True : read delay results from 2_prelimit_simulation/5_RS_saved_results_full/ (full local results, NOT shipped)
#   USE_RAW = False: read delay results from 2_prelimit_simulation/3_RS_saved_results/ (shipped)
#   (the delay_distribution_results CSVs are identical in both; plots are written next to them)
USE_RAW = False
RESULTS_BASE = "5_RS_saved_results_full" if USE_RAW else "3_RS_saved_results"

In [ ]:
# =============================================================================
# delay_distribution_analysis.ipynb
# Reads sim_one_combo outputs and plots delay-time histograms for all 13 policies
# =============================================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# ----- user-adjustable parameters ---------------------------------------------------------
RESULTS_DIR  = Path("../../2_prelimit_simulation/" + RESULTS_BASE + "/3_RS_opt_delay_distribution_5000p/delay_distribution_results")  # directory containing the CSV files
DELAY_SCALE  = 400.0       # delay is divided by this number when plotting
BUCKET_WIDTH = 0.1         # must equal hist_bucket in one_combo_input.txt (sim original unit)
PLOT_BIN_WIDTH = 0.015        # bin width for re-binning when plotting (delay-time unit)
DELAY_BOUND  = 0.33        # position of the dashed "Bound" line in the plot (delay-time unit)
X_LIM        = (0.0, 0.9)  # x axis range of the plot

# ----- new: save directory -------------------------------------------------------
SAVE_DIR = Path("../../2_prelimit_simulation/" + RESULTS_BASE + "/3_RS_opt_delay_distribution_5000p/delay_distribution_plots")  # eps file output directory
SAVE_DIR.mkdir(parents=True, exist_ok=True)

POLICIES = ["proposed", "A1", "A2", "A3", "A4",
            "B1", "B2", "B3", "B4",
            "C1", "C2", "D1", "D2"]

def find_csv(policy: str, suffix: str):
    """match the three naming styles abs_X / rel_X / X (the 5 WMP policies have prefixes, A/B do not)."""
    for prefix in ("abs_", "rel_", ""):
        p = RESULTS_DIR / f"{prefix}{policy}_{suffix}.csv"
        if p.exists():
            return p
    return None

In [ ]:
# =============================================================================
# 1. aggregate 13 _summary.csv files into one table
# =============================================================================
summaries = []
for pol in POLICIES:
    f = find_csv(pol, "summary")
    if f is None:
        print(f"[skip] no summary CSV for {pol}")
        continue
    df = pd.read_csv(f)
    df.insert(0, "filename", f.name)  # add a column for traceability
    summaries.append(df)

summary_table = pd.concat(summaries, ignore_index=True)
print("Summary table:")
display(summary_table)
display(summary_table[['policy', 'cost_mean']])

# optional: save the merged table
# summary_table.to_csv("all_policies_summary.csv", index=False)


In [ ]:
# ----- global font configuration ---------------------------------------------------------
plt.rcParams.update({
    'font.size':       24,
    'text.usetex':     True,
    'axes.titlesize':  26,
    'axes.labelsize':  24,
    'xtick.labelsize': 26,
    'ytick.labelsize': 26,
    'legend.fontsize': 20,
})

# =============================================================================
# 2. draw the Route 1 / Route 2 delay-time histogram for each policy
#    - re-bin by PLOT_BIN_WIDTH when plotting (coarse-grained, visually clear)
#    - 99-quantile and Max are still computed on the original fine-grained data (more accurate)
# =============================================================================

def plot_one_policy(policy: str):
    f = find_csv(policy, "delay_hist")
    if f is None:
        print(f"[skip] no histogram CSV for {policy}")
        return

    df = pd.read_csv(f)
    if df.empty:
        print(f"[skip] empty histogram for {policy}")
        return

    df = df.copy()
    df["delay"] = df["bucket_lower"] / DELAY_SCALE
    fine_bin    = BUCKET_WIDTH / DELAY_SCALE     # original bin width (delay-time unit)

    # --- re-binning: aggregate by PLOT_BIN_WIDTH -----------------------------------
    df["new_bin"] = np.floor(df["delay"] / PLOT_BIN_WIDTH).astype(int)
    rebin = (df.groupby("new_bin", as_index=False)
               .agg(count_route0=("count_route0", "sum"),
                    count_route1=("count_route1", "sum")))
    rebin["bin_lower"] = rebin["new_bin"] * PLOT_BIN_WIDTH

    # --- plotting ----------------------------------------------------------------
    fig, axes = plt.subplots(2, 1, figsize=(6, 5), sharex=True)
    # fig.suptitle(f"Policy: {policy}", fontsize=20)

    for r, ax in enumerate(axes):
        col = f"count_route{r}"

        # coarse-grained (for plotting)
        coarse_counts = rebin[col].values.astype(float)
        coarse_total  = coarse_counts.sum()
        if coarse_total == 0:
            ax.set_title(f"Route {r+1}  (no data)")
            continue
        density = coarse_counts / (coarse_total * PLOT_BIN_WIDTH)

        # fine-grained (for statistics)
        fine_counts = df[col].values.astype(float)
        fine_total  = fine_counts.sum()
        cum = np.cumsum(fine_counts) / fine_total
        q99 = df["delay"].iloc[int(np.searchsorted(cum, 0.99))]
        nonzero = np.flatnonzero(fine_counts > 0)
        max_delay = df["delay"].iloc[nonzero[-1]] + fine_bin

        ax.bar(rebin["bin_lower"], density, width=PLOT_BIN_WIDTH, align="edge",
               color="lightgray", edgecolor="black", linewidth=0.5)
        ax.axvline(DELAY_BOUND, color="black", linestyle=":",  linewidth=2, label="Bound")
        ax.axvline(q99,         color="red",   linestyle="--", linewidth=2, label="99-quantile")
        # ax.axvline(max_delay,   color="blue",  linestyle="-.", linewidth=1.5, label="Max")

        ax.set_title(f"Route {r+1}")
        ax.set_ylabel("Density")
        ax.set_xlim(*X_LIM)
        if r == 0:
            ax.legend(loc="upper right")

        ax.set_xticks([0, 0.2, 0.4, 0.6, 0.8])

    axes[-1].set_xlabel("Delay time")
    plt.tight_layout()

    # ----- save as EPS ------------------------------------------------------
    out_path_eps = SAVE_DIR / f"RS_{policy}_delay_hist.eps"
    out_path_png = SAVE_DIR / f"RS_{policy}_delay_hist.png"
    fig.savefig(out_path_eps, format="eps", bbox_inches="tight")
    fig.savefig(out_path_png, format="png", bbox_inches="tight")
    print(f"Saved: {out_path_eps}")

    plt.show()
    plt.close(fig)        # close the figure to free memory (13 stacked figures take quite a bit)

In [ ]:
# draw all 13 policies
for pol in POLICIES:
    plot_one_policy(pol)

In [ ]:
# =============================================================================
# merge 3 policies into one 2x3 grid figure (Rybko-Stolyar)
# =============================================================================

TARGET_POLICIES = [
    ("proposed", r"Proposed ($\tilde{\tau} = \mathrm{abs}$)"),
    ("A1",       r"Benchmark A1"),
    ("A2",       r"Benchmark A2"),
]

N_ROUTES = 2  # Rybko-Stolyar has only 2 routes

# grid plot uses a small font; use rc_context to temporarily override the global large-font setting
GRID_RC = {
    'font.size':       12,
    'axes.titlesize':  12,
    'axes.labelsize':  12,
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
    'legend.fontsize': 12,
}

X_LIM        = (0.0, 0.65)  # x axis range of the plot


def _prepare_df(policy):
    """load and preprocess the histogram CSV of a single policy."""
    f = find_csv(policy, "delay_hist")
    if f is None:
        return None
    df = pd.read_csv(f)
    if df.empty:
        return None
    df = df.copy()
    df["delay"] = df["bucket_lower"] / DELAY_SCALE
    return df


def plot_combined_grid_rs(policy_list, save_name="combined_delay_hist"):
    # read all data first, exit immediately if any is missing
    dfs, labels = [], []
    for policy, lab in policy_list:
        df = _prepare_df(policy)
        if df is None:
            print(f"[skip] no data for {lab}")
            return
        dfs.append(df)
        labels.append(lab)

    n_cols = len(policy_list)

    with plt.rc_context(GRID_RC):
        fig, axes = plt.subplots(
            N_ROUTES, n_cols,
            figsize=(1.9 * n_cols, 0.8 * N_ROUTES),
            sharex='row',          # same route shares the x axis
            sharey=False,          # each subplot scales y on its own
            gridspec_kw={'hspace': 0.35, 'wspace': 0.22},
        )

        bound_h, q99_h = None, None  # keep a handle for the global legend

        for r in range(N_ROUTES):
            col   = f"count_route{r}"
            bin_w = PLOT_BIN_WIDTH

            for j, df in enumerate(dfs):
                ax = axes[r, j]

                # —— re-bin ——
                tmp = df[["delay", col]].copy()
                tmp["new_bin"] = np.floor(tmp["delay"] / bin_w).astype(int)
                rebin = tmp.groupby("new_bin", as_index=False)[col].sum()
                rebin["bin_lower"] = rebin["new_bin"] * bin_w
                coarse_counts = rebin[col].values.astype(float)
                coarse_total  = coarse_counts.sum()

                if coarse_total > 0:
                    density = coarse_counts / (coarse_total * bin_w)

                    fine_counts = df[col].values.astype(float)
                    fine_total  = fine_counts.sum()
                    cum     = np.cumsum(fine_counts) / fine_total
                    q99_idx = min(int(np.searchsorted(cum, 0.99)), len(df) - 1)
                    q99     = df["delay"].iloc[q99_idx]

                    ax.bar(rebin["bin_lower"], density, width=bin_w, align="edge",
                           color="lightgray", edgecolor="black", linewidth=0.3)
                    bh = ax.axvline(DELAY_BOUND, color="black", linestyle=":",  linewidth=1.2)
                    qh = ax.axvline(q99,         color="red",   linestyle="--", linewidth=1.2)
                    if bound_h is None:
                        bound_h, q99_h = bh, qh

                ax.set_xlim(*X_LIM)
                ax.set_xticks([0.0, 0.2, 0.4, 0.6])

            # hide x tick numbers except for the bottom row
            if r < N_ROUTES - 1:
                for j in range(n_cols):
                    axes[r, j].tick_params(labelbottom=False)

        # column titles: only on the top row
        for j, lab in enumerate(labels):
            axes[0, j].set_title(lab, pad=4)

        # row labels: leftmost column writes Route 1 / Route 2 (horizontal, hugging the left)
        for r in range(N_ROUTES):
            axes[r, 0].set_ylabel(f"Route {r+1}", rotation=0,
                                  ha='right', va='center', labelpad=8)

        # x axis labels only on the bottom row
        for j in range(n_cols):
            axes[N_ROUTES - 1, j].set_xlabel("Delay time")

        fig.subplots_adjust(
            left=0.10, right=0.99,
            top=0.86, bottom=0.14,
            hspace=0.35, wspace=0.22,
        )

        # global legend: at the top-right corner of the whole figure, hugging the axes top edge
        if bound_h is not None:
            fig.legend(
                [bound_h, q99_h], ["Delay bound", "99-quantile"],
                loc='lower center',
                bbox_to_anchor=(0.74, 0.97),
                ncol=2, frameon=False,
            )

        # ----- save ------------------------------------------------------------
        out_eps = SAVE_DIR / f"RS_{save_name}.eps"
        out_pdf = SAVE_DIR / f"RS_{save_name}.pdf"
        out_png = SAVE_DIR / f"RS_{save_name}.png"
        fig.savefig(out_eps, format="eps", bbox_inches="tight")
        fig.savefig(out_pdf, format="pdf", bbox_inches="tight")
        fig.savefig(out_png, format="png", bbox_inches="tight", dpi=300)
        print(f"Saved: {save_name}")
        plt.show()
        plt.close(fig)


plot_combined_grid_rs(TARGET_POLICIES, save_name="combined_delay_hist")